# Step 3 — Generating text

The trained model assigns a probability to every one of 20,000 tokens given what came before.
Turning that into text is a separate decision from training, and the decision matters more than it
looks: the same model produces fluent prose or an infinite loop depending on how its output
distribution is sampled.

In [ ]:
import json, numpy as np, tensorflow as tf

DATA = "../data/processed"; OUT = "../data/outputs"
meta = json.load(open(f"{DATA}/vocab.json"))
itos = meta["itos"]; stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)

trained = tf.keras.models.load_model(f"{OUT}/lstm_lm.keras")
print(f"vocab {VOCAB:,} | trained on sequences of {meta['seq_len']}")

## An inference model that accepts any length

The training model has a fixed 100-token input, because fixed windows are what makes training
efficient. Generation needs the opposite: a context that starts short and grows.

Rather than left-padding every step to 100 — which would feed the model `<pad>` tokens it never saw
in training — the weights are moved into an identical architecture declared with `Input(shape=(None,))`.
An `Embedding`, an `LSTM` and a `Dense` all operate per-timestep, so none of their weight shapes
depend on sequence length; only the declared input does.

In [ ]:
gen_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(None,), dtype="int32"),
    tf.keras.layers.Embedding(VOCAB, trained.layers[0].output_dim),
    tf.keras.layers.LSTM(trained.layers[1].units, return_sequences=True),
    tf.keras.layers.Dense(VOCAB),
])
gen_model.set_weights(trained.get_weights())
print("weights transferred; input length is now unconstrained")

## Reusing the exact preprocessing from Step 1

A seed prompt has to be cleaned and tokenized identically to the training data, or the token ids will
not mean what the model learned. Lowercasing is the one that bites: `"Ông"` and `"ông"` are different
strings, and only the second exists in the vocabulary.

In [ ]:
import re, html, unicodedata

WS    = re.compile(r"\s+")
TOKEN = re.compile(r"[^\W\d_]+|\d+|[^\w\s]", re.UNICODE)

def encode(text):
    t = unicodedata.normalize("NFC", html.unescape(text)).lower()
    return [stoi.get(w, 1) for w in TOKEN.findall(WS.sub(" ", t).strip())]

def decode(ids):
    out = " ".join(itos[i] for i in ids)
    return re.sub(r" ([,.!?;:])", r"\1", out)     # tidy punctuation spacing

seed = "ông ấy nhìn ra ngoài cửa sổ và"
print(encode(seed), "->", decode(encode(seed)))

## Three ways to pick the next token

**Greedy** takes the single highest-probability token every time. It is deterministic, and it is
why untuned generation loops: once the model enters a state whose most likely continuation returns
it to that state, nothing can break the cycle. Language is not the most probable word repeated.

**Temperature** divides the logits before the softmax. Below 1 the distribution sharpens toward the
model's favourites (safer, duller, loops sooner); above 1 it flattens (more surprising, more
incoherent). Temperature 1.0 samples from the model's actual beliefs.

**Top-k** truncates to the k most likely tokens and renormalises, then samples. This is the fix for
temperature's real failure mode: a flattened distribution puts a small but non-zero mass on each of
19,000 nonsense tokens, and their *sum* is large enough that one gets picked regularly. Top-k removes
the tail entirely rather than reweighting it.

In [ ]:
def generate(seed_text, n_tokens=60, temperature=1.0, top_k=None, greedy=False, seed=0):
    rng = np.random.default_rng(seed)
    ids = encode(seed_text)
    for _ in range(n_tokens):
        logits = gen_model.predict(np.array([ids], dtype=np.int32), verbose=0)[0, -1]
        if greedy:
            ids.append(int(logits.argmax())); continue
        logits = logits / temperature
        if top_k:
            cut = np.argpartition(logits, -top_k)[-top_k:]
            masked = np.full_like(logits, -np.inf); masked[cut] = logits[cut]
            logits = masked
        p = np.exp(logits - logits.max()); p /= p.sum()
        ids.append(int(rng.choice(VOCAB, p=p)))
    return decode(ids)

SEED = "ông ấy nhìn ra ngoài cửa sổ và"
print("GREEDY\n ", generate(SEED, 50, greedy=True), "\n")
for t in (0.5, 0.8, 1.2):
    print(f"TEMPERATURE {t}\n ", generate(SEED, 50, temperature=t), "\n")
print("TOP-K 40, temp 0.9\n ", generate(SEED, 50, temperature=0.9, top_k=40))

## Several seeds, one setting

In [ ]:
for s in ["ngày hôm đó trời mưa rất to",
          "cô bé mở cánh cửa và thấy",
          "trong làng có một ông già"]:
    print(f"[{s}]\n  {generate(s, 45, temperature=0.9, top_k=40)}\n")

## Reading the output honestly

What to look for, in the order it appears as a language model learns:

1. **Token-level plausibility** — real Vietnamese words rather than `<unk>` soup. Arrives almost
   immediately, and means nothing on its own.
2. **Local grammar** — classifiers before nouns, `của` between possessor and possessed, punctuation
   in sentence-shaped places. This is what an LSTM over a 100-token window is genuinely good at.
3. **Sentence-level coherence** — a clause that finishes the thought it started.
4. **Cross-sentence coherence** — a character introduced in one sentence still present two sentences
   later. This is where a single-layer LSTM on this much data will fall short, and saying so is more
   useful than pretending otherwise. The cell state is one fixed-size vector carrying everything;
   Transformers replaced LSTMs for exactly this reason.

Comparing against the floor is what keeps the reading honest: uniform guessing over this vocabulary
is a perplexity of 20,000, so the trained model's perplexity says how far from random it is — while
the text above says whether that distance amounts to anything a reader would recognise as Vietnamese.

In [ ]:
metrics = json.load(open(f"{OUT}/lm_metrics.json"))
print(f"val perplexity {metrics['val_perplexity']:,.1f}  vs  uniform {VOCAB:,}")
print(f"trained on {metrics['train_tokens']:,} tokens, {metrics['epochs']} epoch(s), "
      f"{metrics['train_secs']/60:.1f} min")